# Data Merging

### Import

In [2]:

import pandas as pd
from pathlib import Path

### Check the Same

In [6]:
# For each csv, check if have same name for each column

# folder that contains all your CSV files

def check_same(filepath):
    folder = Path(filepath)   # change this

    # get all csv files
    csv_files = sorted(folder.glob("*.csv"))

    if not csv_files:
        print("No CSV files found.")
    else:
        reference_file = csv_files[0]
        reference_cols = pd.read_csv(reference_file, nrows=0).columns.tolist()

        print(f"Reference file: {reference_file.name}")
        print(f"Number of columns: {len(reference_cols)}")
        print("-" * 60)

        all_same = True

        for file in csv_files[1:]:
            cols = pd.read_csv(file, nrows=0).columns.tolist()

            same_count = len(cols) == len(reference_cols)
            same_names = cols == reference_cols

            if same_count and same_names:
                print(f"{file.name}: SAME")
            else:
                all_same = False
                print(f"{file.name}: DIFFERENT")

                if not same_count:
                    print(f"  Column count differs: {len(cols)} vs {len(reference_cols)}")

                if not same_names:
                    print("  Column names/order differ.")

                    missing_in_file = [c for c in reference_cols if c not in cols]
                    extra_in_file = [c for c in cols if c not in reference_cols]

                    if missing_in_file:
                        print("  Missing columns:", missing_in_file)

                    if extra_in_file:
                        print("  Extra columns:", extra_in_file)

                    # check position differences if same set but different order
                    if set(cols) == set(reference_cols) and cols != reference_cols:
                        print("  Same column names, but order is different.")

                print("-" * 60)

        if all_same:
            print("All CSV files have the same number of columns and the same column names in the same order.")
        else:
            print("Some files are different.")
check_same("data/listing")

Reference file: CRMLSListing202401.csv
Number of columns: 84
------------------------------------------------------------
CRMLSListing202402.csv: SAME
CRMLSListing202403.csv: SAME
CRMLSListing202404.csv: SAME
CRMLSListing202405.csv: DIFFERENT
  Column count differs: 82 vs 84
  Column names/order differ.
  Missing columns: ['BuyerAgencyCompensationType', 'BuyerAgencyCompensation']
------------------------------------------------------------
CRMLSListing202406.csv: DIFFERENT
  Column count differs: 82 vs 84
  Column names/order differ.
  Missing columns: ['BuyerAgencyCompensationType', 'BuyerAgencyCompensation']
------------------------------------------------------------
CRMLSListing202407.csv: DIFFERENT
  Column count differs: 82 vs 84
  Column names/order differ.
  Missing columns: ['BuyerAgencyCompensationType', 'BuyerAgencyCompensation']
------------------------------------------------------------
CRMLSListing202408.csv: DIFFERENT
  Column count differs: 82 vs 84
  Column names/orde

### Merge Listing

In [5]:

if not csv_files:
    print("No CSV files found.")
else:
    # use first file as reference
    reference_file = csv_files[0]
    reference_cols = pd.read_csv(reference_file, nrows=0).columns.tolist()

    print(f"Reference file: {reference_file.name}")
    print(f"Reference columns ({len(reference_cols)}):")
    print(reference_cols)
    print("-" * 60)

    merged_dfs = []

    for file in csv_files:
        print(f"Reading {file.name}...")
        df = pd.read_csv(file)

        # keep only columns that are in reference
        df = df.reindex(columns=reference_cols)

        # reindex will:
        # 1. reorder columns to match reference
        # 2. add missing columns as NaN
        # 3. drop extra columns not in reference

        merged_dfs.append(df)

    # stack all rows together
    merged_df = pd.concat(merged_dfs, ignore_index=True)

    print("-" * 60)
    print("Merged shape:", merged_df.shape)
    print("Merged columns:")
    print(merged_df.columns.tolist())

    # save output
    output_file = folder / "mergedListing_24_26.csv"
    merged_df.to_csv(output_file, index=False)

    print(f"Merged file saved to: {output_file}")

Reference file: CRMLSListing202401.csv
Reference columns (84):
['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea', 'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'PropertyType.1', 'MlsStatus', 'ElementarySchool', 'ListAgentFirstName.1', 'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'DaysOnMarket.1', 'BuyerAgencyCompensationType', 'StreetNumberNumeric', 'LivingArea.1', 'ListingId', 'BathroomsTotalInteger', 'City', 'BuyerAgencyCompensation', 'Ta

### Do the same with the Sold, first check the column names.

In [7]:
check_same("data/sold")

Reference file: CRMLSSold202401.csv
Number of columns: 80
------------------------------------------------------------
CRMLSSold202402.csv: DIFFERENT
  Column count differs: 78 vs 80
  Column names/order differ.
  Missing columns: ['BuyerAgentAOR', 'ListAgentAOR', 'OriginatingSystemName', 'OriginatingSystemSubName']
  Extra columns: ['BuyerAgencyCompensationType', 'BuyerAgencyCompensation']
------------------------------------------------------------
CRMLSSold202403.csv: DIFFERENT
  Column count differs: 78 vs 80
  Column names/order differ.
  Missing columns: ['BuyerAgentAOR', 'ListAgentAOR', 'OriginatingSystemName', 'OriginatingSystemSubName']
  Extra columns: ['BuyerAgencyCompensationType', 'BuyerAgencyCompensation']
------------------------------------------------------------
CRMLSSold202404.csv: DIFFERENT
  Column count differs: 78 vs 80
  Column names/order differ.
  Missing columns: ['BuyerAgentAOR', 'ListAgentAOR', 'OriginatingSystemName', 'OriginatingSystemSubName']
  Extra co

In [8]:
def merge_all_csv_full_columns(folder_path, output_name="merged_all_columns.csv"):
    folder = Path(folder_path)
    csv_files = sorted(folder.glob("*.csv"))

    if not csv_files:
        print("No CSV files found.")
        return None

    print(f"Found {len(csv_files)} CSV files.")
    print("-" * 60)

    all_dfs = []
    all_columns = set()

    # first pass: collect all column names
    for file in csv_files:
        try:
            cols = pd.read_csv(file, nrows=0).columns.tolist()
            all_columns.update(cols)
            print(f"{file.name}: {len(cols)} columns found")
        except Exception as e:
            print(f"Error reading columns from {file.name}: {e}")

    # keep column order:
    # first use columns from first file, then append new columns from later files
    first_cols = pd.read_csv(csv_files[0], nrows=0).columns.tolist()
    all_columns_ordered = first_cols.copy()

    for file in csv_files[1:]:
        try:
            cols = pd.read_csv(file, nrows=0).columns.tolist()
            for col in cols:
                if col not in all_columns_ordered:
                    all_columns_ordered.append(col)
        except Exception as e:
            print(f"Error checking column order in {file.name}: {e}")

    print("-" * 60)
    print(f"Total unique columns after union: {len(all_columns_ordered)}")

    # second pass: read each file and align columns
    for file in csv_files:
        try:
            print(f"Reading {file.name} ...")
            df = pd.read_csv(file)

            # align to union of all columns
            df = df.reindex(columns=all_columns_ordered)

            all_dfs.append(df)

        except Exception as e:
            print(f"Error reading {file.name}: {e}")

    if not all_dfs:
        print("No files were successfully read.")
        return None

    merged_df = pd.concat(all_dfs, ignore_index=True)

    output_path = folder / output_name
    merged_df.to_csv(output_path, index=False)

    print("-" * 60)
    print(f"Merged shape: {merged_df.shape}")
    print(f"Saved to: {output_path}")

    return merged_df

merged_df = merge_all_csv_full_columns(r"data/sold",output_name="merged_sold_24-26.csv")

Found 27 CSV files.
------------------------------------------------------------
CRMLSSold202401.csv: 80 columns found
CRMLSSold202402.csv: 78 columns found
CRMLSSold202403.csv: 78 columns found
CRMLSSold202404.csv: 78 columns found
CRMLSSold202405.csv: 80 columns found
CRMLSSold202406.csv: 80 columns found
CRMLSSold202407.csv: 80 columns found
CRMLSSold202408.csv: 78 columns found
CRMLSSold202409.csv: 78 columns found
CRMLSSold202410.csv: 78 columns found
CRMLSSold202411.csv: 78 columns found
CRMLSSold202412.csv: 78 columns found
CRMLSSold202501.csv: 80 columns found
CRMLSSold202502.csv: 78 columns found
CRMLSSold202503.csv: 78 columns found
CRMLSSold202504.csv: 78 columns found
CRMLSSold202505.csv: 78 columns found
CRMLSSold202506.csv: 78 columns found
CRMLSSold202507.csv: 78 columns found
CRMLSSold202508.csv: 78 columns found
CRMLSSold202509.csv: 78 columns found
CRMLSSold202510.csv: 78 columns found
CRMLSSold202511.csv: 78 columns found
CRMLSSold202512.csv: 78 columns found
CRMLSSo

C:\Users\14012\AppData\Local\Temp\ipykernel_91904\2550831624.py:45: DtypeWarning: Columns (0: WaterfrontYN, 1: ElementarySchool, 2: BuilderName, 3: CoBuyerAgentFirstName, 4: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Reading CRMLSSold202405.csv ...
Reading CRMLSSold202406.csv ...
Reading CRMLSSold202407.csv ...
Reading CRMLSSold202408.csv ...
Reading CRMLSSold202409.csv ...
Reading CRMLSSold202410.csv ...
Reading CRMLSSold202411.csv ...
Reading CRMLSSold202412.csv ...
Reading CRMLSSold202501.csv ...
Reading CRMLSSold202502.csv ...
Reading CRMLSSold202503.csv ...
Reading CRMLSSold202504.csv ...
Reading CRMLSSold202505.csv ...
Reading CRMLSSold202506.csv ...


C:\Users\14012\AppData\Local\Temp\ipykernel_91904\2550831624.py:45: DtypeWarning: Columns (0: WaterfrontYN) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Reading CRMLSSold202507.csv ...
Reading CRMLSSold202508.csv ...
Reading CRMLSSold202509.csv ...
Reading CRMLSSold202510.csv ...
Reading CRMLSSold202511.csv ...
Reading CRMLSSold202512.csv ...
Reading CRMLSSold202601.csv ...


C:\Users\14012\AppData\Local\Temp\ipykernel_91904\2550831624.py:45: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


Reading CRMLSSold202602.csv ...
Reading CRMLSSold202603.csv ...
------------------------------------------------------------
Merged shape: (591733, 84)
Saved to: data\sold\merged_sold_24-26.csv


## Filter both to Property Type == "Residential"

In [4]:
df_listing = pd.read_csv("data/mergedListing_24_26.csv")
df_sold = pd.read_csv("data/merged_sold_24_26.csv")


C:\Users\14012\AppData\Local\Temp\ipykernel_109064\2966671545.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listing = pd.read_csv("data/mergedListing_24_26.csv")
C:\Users\14012\AppData\Local\Temp\ipykernel_109064\2966671545.py:2: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv("data/merged_sold_24_26.csv")


### Listing

In [5]:

# row count before Residential filter
print("Listing rows before Residential filter:", df_listing.shape[0])

# keep only Residential
df_listing_residential = df_listing[df_listing["PropertyType"] == "Residential"].copy()

# row count after Residential filter
print("Listing rows after Residential filter:", df_listing_residential.shape[0])

# save filtered listing file
df_listing_residential.to_csv("data/listing.csv", index=False)

Listing rows before Residential filter: 852963
Listing rows after Residential filter: 540183


## Sold

In [6]:

# row count before Residential filter
print("Sold rows before Residential filter:", df_sold.shape[0])

# keep only Residential
df_sold_residential = df_sold[df_sold["PropertyType"] == "Residential"].copy()

# row count after Residential filter
print("Sold rows after Residential filter:", df_sold_residential.shape[0])

# save filtered sold file
df_sold_residential.to_csv("data/sold.csv", index=False)

Sold rows before Residential filter: 591733
Sold rows after Residential filter: 397603
